<a href="https://colab.research.google.com/github/Seomzo/neural-network-challenge-2/blob/main/attrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1: Preprocessing

In [16]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import tensorflow as tf
# from tensorflow.keras.models import Model
# from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [17]:
# Determine the number of unique values in each column
attrition_df.nunique()

,0
Age,43
Attrition,2
BusinessTravel,3
Department,3
DistanceFromHome,29
Education,5
EducationField,6
EnvironmentSatisfaction,4
HourlyRate,71
JobInvolvement,4


In [18]:
# Create y_df with the Attrition and Department columns
y_dep_df = attrition_df[['Department']]
y_att_df = attrition_df[['Attrition']]

y_dep_df.head()

,Department
0,Sales
1,Research & Development
2,Research & Development
3,Research & Development
4,Research & Development


In [19]:
# Create a list of at least 10 column names to use as X data
X_columns = ['Education', 'DistanceFromHome', 'Age', 'JobSatisfaction', 'OverTime', 'StockOptionLevel', 'WorkLifeBalance', 'YearsSinceLastPromotion', 'YearsAtCompany', 'NumCompaniesWorked']

# Create X_df using your selected columns
X_df = attrition_df[X_columns]

# Show the data types for X_df
X_df.dtypes

,0
Education,int64
DistanceFromHome,int64
Age,int64
JobSatisfaction,int64
OverTime,object
StockOptionLevel,int64
WorkLifeBalance,int64
YearsSinceLastPromotion,int64
YearsAtCompany,int64
NumCompaniesWorked,int64


In [20]:
# Convert OverTime to a numeric value
X_df['OverTime'] = X_df['OverTime'].map({'Yes': 1, 'No': 0})
X_df.dtypes


<ipython-input-20-8159bbe899d6>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_df['OverTime'] = X_df['OverTime'].map({'Yes': 1, 'No': 0})


,0
Education,int64
DistanceFromHome,int64
Age,int64
JobSatisfaction,int64
OverTime,int64
StockOptionLevel,int64
WorkLifeBalance,int64
YearsSinceLastPromotion,int64
YearsAtCompany,int64
NumCompaniesWorked,int64


In [21]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_att_train, y_att_test, y_dep_train, y_dep_test = train_test_split(X_df, y_att_df,y_dep_df)

In [22]:
# Create a StandardScaler
scaler = StandardScaler()

# Fit the StandardScaler to the training data
scaler.fit(X_train)

# Scale the training and testing data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [23]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
dep_encoder = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
dep_encoded = dep_encoder.fit_transform(y_dep_df)
dep_columns = dep_encoder.get_feature_names_out()
dep_encoded_df = pd.DataFrame(dep_encoded, columns=dep_columns)
# Create two new variables by applying the encoder
# to the training and testing data
encoded_y_dep_train = dep_encoder.fit_transform(y_dep_train)
encoded_y_dep_test = dep_encoder.fit_transform(y_dep_test)

dep_encoded_df.head()
print(encoded_y_dep_train)

[[0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 ...
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]]


In [24]:
# Create a OneHotEncoder for the Attrition column
att_encoder = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
att_encoder.fit(y_att_train)

# Create two new variables by applying the encoder
# to the training and testing data
encoded_y_att_train = att_encoder.transform(y_att_train)
encoded_y_att_test = att_encoder.transform(y_att_test)

encoded_y_att_train

array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [1., 0.],
       [1., 0.],
       [1., 0.]])

## Part 2: Create, Compile, and Train the Model

In [25]:
# Find the number of columns in the X training data.
num_cols = len(X_train.columns)

# Create the input layer
input_layer = tf.keras.layers.Input(shape=(num_cols,), name='input_layer')

# Create at least two shared layers
shared_layer_1 = tf.keras.layers.Dense(units=10, activation='relu')(input_layer)
shared_layer_2 = tf.keras.layers.Dense(units=10, activation='relu')(shared_layer_1)


In [26]:
# Create a branch for Department
# with a hidden layer and an output layer

# Create the hidden layer
hidden_layer = tf.keras.layers.Dense(units=10, activation='relu')(shared_layer_2)

# Create the output layer
dep_output = tf.keras.layers.Dense(units=3, activation='softmax',name='department_output')(hidden_layer)

In [27]:
# Create a branch for Attrition
# with a hidden layer and an output layer

# Create the hidden layer
hidden_layer2 = tf.keras.layers.Dense(units=10, activation='relu')(shared_layer_2)

# Create the output layer
attrition_output = tf.keras.layers.Dense(units=2, activation='sigmoid',name='attrition_output')(hidden_layer2)

In [28]:
# Create the model
model = tf.keras.models.Model(inputs=input_layer, outputs=[dep_output, attrition_output])

# Compile the model
model.compile(optimizer='adam',
              loss={'department_output': 'categorical_crossentropy', 'attrition_output': 'binary_crossentropy'},
              metrics={'department_output':'accuracy', 'attrition_output':'accuracy'})

# Summarize the model
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 10)             │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_5 (Dense)           │ (None, 10)             │            110 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_6 (Dense)           │ (None, 10)             │            110 │ dense_5[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_7 (Dense)           │ (None, 10)             │            110 │ dense_6[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_8 (Dense)           │ (None, 10)             │            110 │ dense_6[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ department_output (Dense) │ (None, 3)              │             33 │ dense_7[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attrition_output (Dense)  │ (None, 2)              │             22 │ dense_8[0][0]          │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 495 (1.93 KB)

 Trainable params: 495 (1.93 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Train the model
fit_model = model.fit(X_train_scaled, {'department_output': encoded_y_dep_train, 'attrition_output': encoded_y_att_train}, epochs=100, batch_size=32, validation_split=0.2)

Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - attrition_output_accuracy: 0.3081 - attrition_output_loss: 0.7720 - department_output_accuracy: 0.0492 - department_output_loss: 1.3272 - loss: 2.0994 - val_attrition_output_accuracy: 0.5701 - val_attrition_output_loss: 0.7035 - val_department_output_accuracy: 0.2805 - val_department_output_loss: 1.1194 - val_loss: 1.8224
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - attrition_output_accuracy: 0.6372 - attrition_output_loss: 0.6890 - department_output_accuracy: 0.4076 - department_output_loss: 1.0791 - loss: 1.7682 - val_attrition_output_accuracy: 0.8235 - val_attrition_output_loss: 0.6336 - val_department_output_accuracy: 0.6244 - val_department_output_loss: 0.9907 - val_loss: 1.6237
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - attrition_output_accuracy: 0.8328 - attrition_output_loss: 0.6313 - department_output_accuracy: 0.6384 - department_output_loss: 0.9660 - loss: 1.5974 - val_attrition_output_accuracy: 0.8552 -

In [30]:
# Evaluate the model with the testing data
test_results = model.evaluate(X_test_scaled, {'department_output': encoded_y_dep_test, 'attrition_output': encoded_y_att_test})
test_results

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - attrition_output_accuracy: 0.8501 - attrition_output_loss: 0.3803 - department_output_accuracy: 0.6095 - department_output_loss: 0.8635 - loss: 1.2422 


[1.3157036304473877,
 0.8907921314239502,
 0.4356098175048828,
 0.8396739363670349,
 0.595108687877655]

In [32]:
# Print the accuracy for both department and attrition
print(f"Department Accuracy: {test_results[3]}")
print(f"Attrition Accuracy: {test_results[4]}")

Department Accuracy: 0.8396739363670349
Attrition Accuracy: 0.595108687877655


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. accuracy seems like it would be the best metric
2. Softmax for the department output due to having 3 outputs, and sigmoid for the attrition because it's binary.
3. more data.